# Matchmaking: De Probabilidades a Decisiones

Notebook de soporte para la charla en UdeSA. Pipeline completo:
1. Generación de datos sintéticos
2. Estimación de probabilidades
3. Construcción del grafo de matching
4. Algoritmos de matching
5. Comparación de resultados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy.special import expit  # sigmoide
from scipy.optimize import linear_sum_assignment, minimize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.calibration import calibration_curve
from matching.games import StableMarriage
from pathlib import Path
import time

np.random.seed(42)
IMG_DIR = Path("img")
IMG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
})

---
## Sección 1: Generación de datos sintéticos

100 usuarios con 6 dimensiones cada uno:
- Edad (normalizada), extroversión, deportividad, intelectualidad, aventura, romanticismo
- Todas en [0, 1]

Cada usuario tiene:
- **Vector de características**: lo que "es"
- **Vector de preferencias**: lo que busca

In [ ]:
N_USERS = 100
N_DIMS = 6
DIM_NAMES = ["edad", "extroversion", "deportividad", "intelectualidad", "aventura", "romanticismo"]

# Vectores de características y preferencias (uniformes en [0,1])
features = np.random.rand(N_USERS, N_DIMS)
preferences = np.random.rand(N_USERS, N_DIMS)

# Peso que cada usuario le da a cada dimensión (suman 1)
raw_weights = np.random.rand(N_USERS, N_DIMS)
pref_weights = raw_weights / raw_weights.sum(axis=1, keepdims=True)

print(f"Usuarios: {N_USERS}, Dimensiones: {N_DIMS}")
print(f"\nEjemplo — Usuario 0:")
print(f"  Características: {features[0].round(2)}")
print(f"  Preferencias:    {preferences[0].round(2)}")
print(f"  Pesos:           {pref_weights[0].round(2)}")

In [ ]:
def p_directional(i, j, features, preferences, pref_weights):
    """P(i -> j): probabilidad de que i apruebe a j.
    
    Basada en la distancia ponderada entre lo que i busca y lo que j ofrece,
    pasada por una sigmoide.
    """
    diff = preferences[i] - features[j]
    weighted_dist = np.sum(pref_weights[i] * diff**2)
    # Sigmoide centrada: distancia 0 -> prob ~0.73, distancia alta -> prob baja
    return expit(2 - 5 * weighted_dist)


def p_mutual_true(i, j, features, preferences, pref_weights, correlation=0.15):
    """P(i <-> j): probabilidad mutua real (ground truth).
    
    NO es simplemente el producto de las direccionales.
    Incluye un término de correlación: la gente que ofrece lo que otros
    buscan tiende a buscar cosas similares.
    """
    p_ij = p_directional(i, j, features, preferences, pref_weights)
    p_ji = p_directional(j, i, features, preferences, pref_weights)
    product = p_ij * p_ji
    # Término de correlación positiva
    corr_term = correlation * min(p_ij, p_ji)
    return np.clip(product + corr_term, 0, 1)


# Calcular todas las probabilidades
P_dir = np.zeros((N_USERS, N_USERS))
P_mut = np.zeros((N_USERS, N_USERS))

for i in range(N_USERS):
    for j in range(N_USERS):
        if i != j:
            P_dir[i, j] = p_directional(i, j, features, preferences, pref_weights)
            if i < j:
                P_mut[i, j] = p_mutual_true(i, j, features, preferences, pref_weights)
                P_mut[j, i] = P_mut[i, j]

# Comparar P_mutua real vs producto de direccionales
P_product = np.zeros((N_USERS, N_USERS))
for i in range(N_USERS):
    for j in range(i+1, N_USERS):
        P_product[i, j] = P_dir[i, j] * P_dir[j, i]
        P_product[j, i] = P_product[i, j]

print(f"P direccional media: {P_dir[P_dir > 0].mean():.3f}")
print(f"P mutua real media: {P_mut[P_mut > 0].mean():.3f}")
print(f"P producto media: {P_product[P_product > 0].mean():.3f}")

In [ ]:
# Generar observaciones históricas (accept/reject)
# Simulamos que se mostraron ~20 candidatos aleatorios a cada usuario
N_SHOWN = 20
observations = []

for i in range(N_USERS):
    candidates = np.random.choice(
        [j for j in range(N_USERS) if j != i], size=N_SHOWN, replace=False
    )
    for j in candidates:
        p = p_directional(i, j, features, preferences, pref_weights)
        accepted = np.random.rand() < p
        observations.append({
            "user_i": i,
            "user_j": j,
            "accepted": int(accepted),
            "p_true": p,
        })

df_obs = pd.DataFrame(observations)
print(f"Observaciones generadas: {len(df_obs)}")
print(f"Tasa de aceptación: {df_obs['accepted'].mean():.1%}")
df_obs.head()

---
## 🎯 PAUSA CHARLA
Volver a slides para explicar el problema conceptualmente (probabilidades direccionales vs. mutuas, el dilema de Brad Pitt / Penélope Cruz).

---
## Sección 2: Estimación de probabilidades

Dos modelos:
1. **Logistic Regression**: interpretable
2. **Gradient Boosted Trees**: más preciso, menos interpretable

In [ ]:
# Construir features para los modelos
# Para cada par (i, j): diferencia e interacción entre preferencias de i y características de j

def build_pair_features(user_i, user_j):
    """Features para predecir si user_i aprueba a user_j."""
    diff = preferences[user_i] - features[user_j]
    interaction = preferences[user_i] * features[user_j]
    return np.concatenate([diff, interaction, pref_weights[user_i]])


X = np.array([build_pair_features(row.user_i, row.user_j) for _, row in df_obs.iterrows()])
y = df_obs["accepted"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
idx_train, idx_test = train_test_split(range(len(df_obs)), test_size=0.3, random_state=42)

print(f"Features por par: {X.shape[1]} (6 diff + 6 interact + 6 weights)")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_prob_lr = lr.predict_proba(X_test)[:, 1]
auc_lr = roc_auc_score(y_test, y_prob_lr)

# Gradient Boosted Trees
gbt = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
gbt.fit(X_train, y_train)
y_prob_gbt = gbt.predict_proba(X_test)[:, 1]
auc_gbt = roc_auc_score(y_test, y_prob_gbt)

print(f"AUC Logistic Regression: {auc_lr:.3f}")
print(f"AUC Gradient Boosted Trees: {auc_gbt:.3f}")

In [ ]:
# Coeficientes del modelo logístico (interpretabilidad)
feature_names = (
    [f"diff_{d}" for d in DIM_NAMES]
    + [f"inter_{d}" for d in DIM_NAMES]
    + [f"weight_{d}" for d in DIM_NAMES]
)
coef_df = pd.DataFrame({"feature": feature_names, "coef": lr.coef_[0]})
coef_df = coef_df.reindex(coef_df["coef"].abs().sort_values(ascending=True).index)

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(coef_df["feature"], coef_df["coef"], color="steelblue")
ax.set_xlabel("Coeficiente")
ax.set_title("Logistic Regression — Coeficientes")
ax.grid(True, alpha=0.3, axis="x")
fig.tight_layout()
plt.show()

In [ ]:
# Curvas de calibración
fig, ax = plt.subplots(figsize=(8, 8))

for name, y_prob in [("Logistic Regression", y_prob_lr), ("GBT", y_prob_gbt)]:
    prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)
    ax.plot(prob_pred, prob_true, "o-", label=name)

ax.plot([0, 1], [0, 1], "--", color="gray", label="Calibración perfecta")
ax.set_xlabel("Probabilidad predicha")
ax.set_ylabel("Proporción real de aceptaciones")
ax.set_title("Curvas de calibración")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Estimar P_mutua: producto de direccionales vs. realidad
# Usamos GBT como nuestro mejor modelo para estimar P_dir

def estimate_p_directional(model, i, j):
    """Estimar P(i -> j) usando el modelo entrenado."""
    x = build_pair_features(i, j).reshape(1, -1)
    return model.predict_proba(x)[0, 1]


# Estimar P_mutua como producto para todos los pares
P_mut_est = np.zeros((N_USERS, N_USERS))
for i in range(N_USERS):
    for j in range(i+1, N_USERS):
        p_ij = estimate_p_directional(gbt, i, j)
        p_ji = estimate_p_directional(gbt, j, i)
        P_mut_est[i, j] = p_ij * p_ji
        P_mut_est[j, i] = P_mut_est[i, j]

# Comparar con ground truth
mask = np.triu(np.ones((N_USERS, N_USERS), dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(P_mut[mask], P_mut_est[mask], alpha=0.3, s=10)
ax.plot([0, 1], [0, 1], "--", color="red")
ax.set_xlabel("P mutua real")
ax.set_ylabel("P mutua estimada (producto)")
ax.set_title("P mutua real vs. estimada como producto de direccionales")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

corr = np.corrcoef(P_mut[mask], P_mut_est[mask])[0, 1]
print(f"Correlación: {corr:.3f}")
print(f"Sesgo medio (est - real): {(P_mut_est[mask] - P_mut[mask]).mean():.4f}")

---
## Sección 3: Grafo de matching

Construimos el grafo completo ponderado de los 100 usuarios.
Peso de cada arista = P_mutua estimada.

In [ ]:
# Construir grafo
G = nx.Graph()
G.add_nodes_from(range(N_USERS))

for i in range(N_USERS):
    for j in range(i+1, N_USERS):
        if P_mut_est[i, j] > 0.01:  # filtrar aristas muy débiles para visualización
            G.add_edge(i, j, weight=P_mut_est[i, j])

print(f"Nodos: {G.number_of_nodes()}")
print(f"Aristas (peso > 0.01): {G.number_of_edges()}")

In [ ]:
# Visualización de subgrafo (~20 nodos con mejores probabilidades mutuas)
# Elegimos los 20 nodos con mayor suma de P_mutua
node_strength = P_mut_est.sum(axis=1)
top_nodes = np.argsort(node_strength)[-20:]

G_sub = G.subgraph(top_nodes).copy()
weights = [G_sub[u][v]["weight"] for u, v in G_sub.edges()]
max_w = max(weights) if weights else 1

fig, ax = plt.subplots(figsize=(12, 12))
pos = nx.spring_layout(G_sub, seed=42, k=2)
nx.draw_networkx_nodes(G_sub, pos, node_size=300, node_color="lightblue", ax=ax)
nx.draw_networkx_labels(G_sub, pos, font_size=9, ax=ax)
nx.draw_networkx_edges(
    G_sub, pos,
    width=[3 * w / max_w for w in weights],
    alpha=[0.3 + 0.7 * w / max_w for w in weights],
    edge_color="steelblue",
    ax=ax,
)
ax.set_title("Subgrafo: 20 usuarios con mayor P mutua total")
ax.axis("off")
fig.tight_layout()
plt.show()

---
## 🎯 PAUSA CHARLA
Volver a slides para explicar los algoritmos de matching antes de correrlos (Gale-Shapley, max weight, fuerza bruta).

---
## Sección 4: Algoritmos de matching

### 4a. Gale-Shapley (Stable Matching)

Rankings derivados de las probabilidades direccionales estimadas.
Garantiza estabilidad, no bienestar global.

In [ ]:
# Dividir usuarios en dos grupos para Gale-Shapley (requiere dos "lados")
group_a = list(range(0, N_USERS // 2))
group_b = list(range(N_USERS // 2, N_USERS))

# Construir rankings: cada usuario en grupo_a rankea a todos en grupo_b y viceversa
# Ranking basado en P_direccional estimada
prefs_a = {}
for i in group_a:
    scores = [(j, estimate_p_directional(gbt, i, j)) for j in group_b]
    scores.sort(key=lambda x: -x[1])
    prefs_a[i] = [j for j, _ in scores]

prefs_b = {}
for j in group_b:
    scores = [(i, estimate_p_directional(gbt, j, i)) for i in group_a]
    scores.sort(key=lambda x: -x[1])
    prefs_b[j] = [i for i, _ in scores]

print(f"Grupo A: {len(group_a)} usuarios, Grupo B: {len(group_b)} usuarios")
print(f"Ejemplo ranking de usuario 0: {prefs_a[0][:5]}... (top 5)")

In [ ]:
# Implementación propia de Gale-Shapley
def gale_shapley(prefs_proposers, prefs_reviewers):
    """Gale-Shapley con el primer grupo como proponentes."""
    free = list(prefs_proposers.keys())
    proposals = {p: 0 for p in free}  # índice del próximo a proponer
    matches = {}  # reviewer -> proposer
    
    # Pre-computar ranking inverso para reviewers
    rank = {}
    for r, pref_list in prefs_reviewers.items():
        rank[r] = {p: i for i, p in enumerate(pref_list)}
    
    while free:
        p = free.pop(0)
        r = prefs_proposers[p][proposals[p]]
        proposals[p] += 1
        
        if r not in matches:
            matches[r] = p
        elif rank[r][p] < rank[r][matches[r]]:
            # r prefiere a p sobre su match actual
            free.append(matches[r])
            matches[r] = p
        else:
            free.append(p)
    
    return {v: k for k, v in matches.items()}  # proposer -> reviewer


t0 = time.time()
gs_matching = gale_shapley(prefs_a, prefs_b)
t_gs = time.time() - t0

# Peso total del matching GS
gs_weight = sum(P_mut_est[i, j] for i, j in gs_matching.items())
print(f"Gale-Shapley: {len(gs_matching)} pares, peso total = {gs_weight:.3f}, tiempo = {t_gs:.4f}s")

In [ ]:
# Verificación con librería `matching`
game = StableMarriage.create_from_dictionaries(prefs_a, prefs_b)
lib_result = game.solve()
gs_matching_lib = {p.name: r.name for p, r in lib_result.items()}

# Comparar
gs_weight_lib = sum(P_mut_est[i, j] for i, j in gs_matching_lib.items())
print(f"Librería matching: {len(gs_matching_lib)} pares, peso total = {gs_weight_lib:.3f}")
print(f"Implementación propia y librería coinciden: {gs_matching == gs_matching_lib}")

### 4b. Maximum Weight Matching

Usando `networkx.max_weight_matching` y alternativamente `scipy.optimize.linear_sum_assignment`.

In [ ]:
# networkx max weight matching (sobre el grafo completo, no bipartito)
G_full = nx.Graph()
G_full.add_nodes_from(range(N_USERS))
for i in range(N_USERS):
    for j in range(i+1, N_USERS):
        G_full.add_edge(i, j, weight=P_mut_est[i, j])

t0 = time.time()
mwm_matching = nx.max_weight_matching(G_full, maxcardinality=True)
t_mwm = time.time() - t0

mwm_weight = sum(P_mut_est[i, j] for i, j in mwm_matching)
print(f"Max Weight Matching: {len(mwm_matching)} pares, peso total = {mwm_weight:.3f}, tiempo = {t_mwm:.4f}s")

In [ ]:
# Alternativa: scipy linear_sum_assignment (formulación como problema de asignación)
# Nota: solo funciona para matching bipartito, lo usamos sobre los mismos grupos
cost_matrix = np.zeros((len(group_a), len(group_b)))
for ii, i in enumerate(group_a):
    for jj, j in enumerate(group_b):
        cost_matrix[ii, jj] = -P_mut_est[i, j]  # negativo porque minimiza

t0 = time.time()
row_ind, col_ind = linear_sum_assignment(cost_matrix)
t_scipy = time.time() - t0

scipy_matching = {group_a[ii]: group_b[jj] for ii, jj in zip(row_ind, col_ind)}
scipy_weight = sum(P_mut_est[i, j] for i, j in scipy_matching.items())
print(f"Scipy (bipartito): {len(scipy_matching)} pares, peso total = {scipy_weight:.3f}, tiempo = {t_scipy:.4f}s")

### 4c. Fuerza bruta con scipy.minimize

Formulación como optimización continua relajada con límite de iteraciones.

In [ ]:
# Fuerza bruta: relajación continua del problema de asignación
# Variables: matriz X de N/2 x N/2 (bipartita), X[i,j] en [0,1]
# Objetivo: maximizar sum(X[i,j] * P_mut_est[group_a[i], group_b[j]])
# Restricciones: filas y columnas suman <= 1

n_a, n_b = len(group_a), len(group_b)
P_matrix = np.array([[P_mut_est[i, j] for j in group_b] for i in group_a])


def objective(x):
    X = x.reshape(n_a, n_b)
    return -np.sum(X * P_matrix)  # negativo para minimizar


# Restricciones: filas y columnas suman <= 1
constraints = []
for i in range(n_a):
    constraints.append({"type": "ineq", "fun": lambda x, i=i: 1 - x.reshape(n_a, n_b)[i, :].sum()})
for j in range(n_b):
    constraints.append({"type": "ineq", "fun": lambda x, j=j: 1 - x.reshape(n_a, n_b)[:, j].sum()})

bounds = [(0, 1)] * (n_a * n_b)
x0 = np.full(n_a * n_b, 1.0 / max(n_a, n_b))  # initial guess uniforme

t0 = time.time()
result = minimize(objective, x0, method="SLSQP", bounds=bounds, constraints=constraints,
                  options={"maxiter": 1000, "ftol": 1e-9})
t_brute = time.time() - t0

# Redondear a matching discreto (asignación greedy)
X_opt = result.x.reshape(n_a, n_b)
bf_matching = {}
used_b = set()
for ii in np.argsort(-X_opt.max(axis=1)):  # empezar por la fila con mayor valor
    for jj in np.argsort(-X_opt[ii]):
        if jj not in used_b:
            bf_matching[group_a[ii]] = group_b[jj]
            used_b.add(jj)
            break

bf_weight = sum(P_mut_est[i, j] for i, j in bf_matching.items())
print(f"Fuerza bruta: {len(bf_matching)} pares, peso total = {bf_weight:.3f}, tiempo = {t_brute:.4f}s")
print(f"Convergió: {result.success}, iteraciones: {result.nit}")

---
## Sección 5: Comparación de resultados

In [ ]:
# Tabla comparativa
results = pd.DataFrame({
    "Algoritmo": ["Gale-Shapley", "Max Weight (networkx)", "Scipy bipartito", "Fuerza bruta (SLSQP)"],
    "Pares": [len(gs_matching), len(mwm_matching), len(scipy_matching), len(bf_matching)],
    "Peso total": [gs_weight, mwm_weight, scipy_weight, bf_weight],
    "Tiempo (s)": [t_gs, t_mwm, t_scipy, t_brute],
})
results["% del óptimo (bipartito)"] = (results["Peso total"] / scipy_weight * 100).round(1)
results

In [ ]:
# Visualización: comparar matchings en el subgrafo
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

matchings = {
    "Gale-Shapley": set((min(i,j), max(i,j)) for i, j in gs_matching.items()),
    "Max Weight": set((min(i,j), max(i,j)) for i, j in mwm_matching),
    "Fuerza bruta": set((min(i,j), max(i,j)) for i, j in bf_matching.items()),
}

# Usar los mismos 20 nodos del subgrafo
top_set = set(top_nodes)

for ax, (name, match_set) in zip(axes, matchings.items()):
    # Filtrar matching a nodos del subgrafo
    match_in_sub = [(i, j) for i, j in match_set if i in top_set and j in top_set]
    
    nx.draw_networkx_nodes(G_sub, pos, node_size=200, node_color="lightblue", ax=ax)
    nx.draw_networkx_labels(G_sub, pos, font_size=8, ax=ax)
    
    # Aristas del grafo en gris claro
    nx.draw_networkx_edges(G_sub, pos, alpha=0.1, edge_color="gray", ax=ax)
    
    # Aristas del matching en color
    if match_in_sub:
        nx.draw_networkx_edges(
            G_sub, pos, edgelist=match_in_sub,
            width=3, edge_color="red", alpha=0.8, ax=ax,
        )
    
    weight_sub = sum(P_mut_est[i, j] for i, j in match_in_sub)
    ax.set_title(f"{name}\n(peso subgrafo: {weight_sub:.3f})")
    ax.axis("off")

fig.suptitle("Comparación de matchings (subgrafo de 20 nodos)", fontsize=16)
fig.tight_layout()
fig.savefig(IMG_DIR / "matching_comparison.svg", bbox_inches="tight")
plt.show()
print("✓ Guardado en img/matching_comparison.svg")

In [ ]:
# Insight: ¿el par con mayor P mutua está en cada matching?
# Encontrar el par con mayor P_mutua estimada
best_i, best_j = np.unravel_index(np.argmax(P_mut_est), P_mut_est.shape)
best_p = P_mut_est[best_i, best_j]
print(f"Par con mayor P mutua estimada: ({best_i}, {best_j}) = {best_p:.4f}")
print()

pair = (min(best_i, best_j), max(best_i, best_j))
for name, match_set in matchings.items():
    present = pair in match_set
    print(f"  {name}: {'✓ incluido' if present else '✗ NO incluido'}")

In [ ]:
# Guardar tabla como imagen para slides de respaldo
fig, ax = plt.subplots(figsize=(12, 3))
ax.axis("off")
table = ax.table(
    cellText=results.values,
    colLabels=results.columns,
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(14)
table.scale(1.2, 1.8)
fig.tight_layout()
fig.savefig(IMG_DIR / "matching_results_table.svg", bbox_inches="tight")
plt.show()
print("✓ Guardado en img/matching_results_table.svg")